# Chapter 9, Guide 1: QAOA with a fixed adiabatic schedule

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 9,
"Building QAOA" and "Testing QAOA", on the book's five-vertex weighted MaxCut instance.

The six steps: (1) build $Q$ and $\mathbf c$ classically and test against the cost
function on all $2^n$ strings, then negate once for a maximization; (2) build $H_C$ as a
list of (Pauli string, coefficient) pairs and check its diagonal; (3) the cost layer
through `pauli_evolution`, with the factor of two; (4) the mixer $R_x(-2\beta)$;
(5) the schedule $\gamma_j=(j/p)\Delta t$, $\beta_j=(1-j/p)\Delta t$, $\Delta t=T/p$;
(6) sample, evaluate $C(\mathbf x)$ classically, report the best.

Convention: book wire $j$ (top, most significant) is Qiskit qubit `bq(j)`; bit $x_j$ of
the assignment lives on wire $j$.

In [1]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

bq = lambda j, tot: tot - 1 - j


def pauli_evolution(circuit, pauli_string, angle, qubits):
    active = [k for k, p in enumerate(pauli_string) if p != "I"]
    if not active:
        return
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.sdg(qubits[k]); circuit.h(qubits[k])
    last = qubits[active[-1]]
    for k in active[:-1]: circuit.cx(qubits[k], last)
    circuit.rz(2 * angle, last)
    for k in reversed(active[:-1]): circuit.cx(qubits[k], last)
    for k in active:
        if pauli_string[k] == "X": circuit.h(qubits[k])
        elif pauli_string[k] == "Y": circuit.h(qubits[k]); circuit.s(qubits[k])

## Step 1: the instance, $Q$ and $\mathbf c$, tested on all 32 strings

In [2]:
W = np.array([[0, 2, 0, 1, 1],
              [2, 0, 2, 0, 0],
              [0, 2, 0, 2, 0],
              [1, 0, 2, 0, 3],
              [1, 0, 0, 3, 0]], dtype=float)
n = W.shape[0]


def cut_weight(x):
    return sum(W[i, j] * x[i] * (1 - x[j]) for i in range(n) for j in range(n))


Q = -W
c = W.sum(axis=1)
strings = [np.array(x) for x in itertools.product([0, 1], repeat=n)]
for x in strings:
    assert abs(x @ Q @ x + c @ x - cut_weight(x)) < 1e-12
C_max = max(cut_weight(x) for x in strings)
best = [x for x in strings if cut_weight(x) == C_max]
print(f"QUBO form checked on all {2**n} strings. Max cut = {C_max}, attained by {[''.join(map(str, x)) for x in best]}")
print(f"Total edge weight = {W.sum()/2}, random-string average cut = {W.sum()/4}")

# maximization: negate once, here and nowhere else
Q_min, c_min = -Q, -c

QUBO form checked on all 32 strings. Max cut = 10.0, attained by ['01010', '10101']
Total edge weight = 11.0, random-string average cut = 5.5


## Step 2: $H_C$ as Pauli terms, diagonal checked against $-C(\mathbf x)$

In [3]:
def ising_terms(Q, c):
    """H_C = sum_{ij} Q_ij/4 Z_i Z_j - sum_i (c_i + sum_j Q_ij)/2 Z_i + const, Q symmetric.
    Returns (zz_terms as {(i,j): coeff over unordered pairs}, z_terms {i: coeff}, const)."""
    n = len(c)
    zz, z = {}, {}
    const = Q.sum() / 4 + c.sum() / 2
    for i in range(n):
        for j in range(i + 1, n):
            zz[(i, j)] = (Q[i, j] + Q[j, i]) / 4              # the ordered pairs (i,j) and (j,i) together
        z[i] = -0.5 * (c[i] + Q[i].sum())
    # the i == j terms of the first sum are constants (Z_i^2 = I) and sit inside Q.sum()/4
    return zz, z, const


zz, z, const = ising_terms(Q_min, c_min)


def label(pairs):
    s = ["I"] * n
    for q, L in pairs: s[q] = L
    return "".join(s)


terms = [(label([(i, "Z"), (j, "Z")]), v) for (i, j), v in zz.items() if abs(v) > 1e-12]
terms += [(label([(i, "Z")]), v) for i, v in z.items() if abs(v) > 1e-12]
H_C = SparsePauliOp.from_list(terms + [("I" * n, const)])
diag = np.real(np.diag(H_C.to_matrix()))
for x in strings:
    idx = int("".join(map(str, x)), 2)              # wire 0 = most significant bit
    assert abs(diag[idx] + cut_weight(x)) < 1e-12
print("diag(H_C) = -C(x) on all 32 strings (sign flipped for maximization)")

diag(H_C) = -C(x) on all 32 strings (sign flipped for maximization)


## Steps 3 to 5: the circuit

In [4]:
q = [bq(j, n) for j in range(n)]


def qaoa_circuit(p, T, measure=False):
    dt = T / p
    qc = QuantumCircuit(n, n if measure else 0)
    qc.h(q)
    for j in range(1, p + 1):
        gamma, beta = (j / p) * dt, (1 - j / p) * dt
        for P, coef in terms:                        # cost layer: R_z(2 * coef * gamma) per term
            pauli_evolution(qc, P, coef * gamma, q)
        qc.rx(-2 * beta, q)                          # mixer: H_M = -sum X_i, beta > 0
    if measure:
        qc.measure(q, [n - 1 - j for j in range(n)])
    return qc


def approximation_ratio(p, T):
    sv = Statevector(qaoa_circuit(p, T))
    probs = sv.probabilities(q[::-1])                # indexed by the book integer of x
    expect = sum(pr * cut_weight(x) for pr, x in zip(probs, strings))
    return expect / C_max

## Tests

1. $p=1$: $\beta_1=0$, the circuit is $|+\rangle^{\otimes n}$ measured, ratio $5.5/10=0.550$.
2. $U_C(\gamma)$ is diagonal with entries $e^{-i\gamma C(\mathbf x)}$ up to the dropped identity.
3. At $T=40$ the book quotes $0.550, 0.667, 0.531, 0.618, 0.660, 0.491$ for
   $p=1,2,4,8,16,32$ and $1.000$ at $p=64$. A dip is not a bug.
4. Compare the best sample against brute force.

In [5]:
# Test 1
r1 = approximation_ratio(1, 40)
print(f"p=1, T=40: ratio {r1:.3f} (expect 0.550)")
assert abs(r1 - 0.55) < 1e-9

# Test 2
gamma = 0.37
qcC = QuantumCircuit(n)
for P, coef in terms: pauli_evolution(qcC, P, coef * gamma, q)
from qiskit.quantum_info import Operator
UC = Operator(qcC).data
assert np.allclose(UC, np.diag(np.diag(UC)))
phases = np.angle(np.diag(UC) * np.exp(-1j * gamma * const))      # restore the dropped identity
expected = np.array([-gamma * (-cut_weight(x)) for x in strings])
assert np.allclose(np.exp(1j * phases), np.exp(1j * expected))
print("U_C(gamma) is diagonal with entries exp(-i gamma (-C(x))) up to the identity phase")

# Test 3
T = 40
ratios = {p: approximation_ratio(p, T) for p in [1, 2, 4, 8, 16, 32, 64]}
book = {1: 0.550, 2: 0.667, 4: 0.531, 8: 0.618, 16: 0.660, 32: 0.491, 64: 1.000}
for p, r in ratios.items():
    print(f"p={p:3d}: ratio {r:.3f}   book {book[p]:.3f}")
    assert abs(r - book[p]) < 2e-3
plt.semilogx(list(ratios), list(ratios.values()), "o-", base=2)
plt.xlabel("p"); plt.ylabel("approximation ratio at T=40"); plt.show()

p=1, T=40: ratio 0.550 (expect 0.550)
U_C(gamma) is diagonal with entries exp(-i gamma (-C(x))) up to the identity phase
p=  1: ratio 0.550   book 0.550
p=  2: ratio 0.667   book 0.667
p=  4: ratio 0.531   book 0.531
p=  8: ratio 0.618   book 0.618
p= 16: ratio 0.660   book 0.660
p= 32: ratio 0.491   book 0.491
p= 64: ratio 1.000   book 1.000


## Scanning $T$: the ratio rises toward 1 as $T$ grows, until $\Delta t$ gets too large

At fixed $p$ a larger $T$ means a larger $\Delta t=T/p$, and the Trotterized adiabatic
step eventually breaks down: that is the collapse at $T=80$, $p=64$. Scan $T$ and $p$
together, as the guide says, and watch the ratio saturate.

In [6]:
for T_ in [5, 10, 20, 40, 80]:
    print(f"T={T_:3d}, p=64, dt={T_/64:.3f}: ratio {approximation_ratio(64, T_):.3f}")
print(f"T= 80, p=128, dt={80/128:.3f}: ratio {approximation_ratio(128, 80):.3f}")

T=  5, p=64, dt=0.078: ratio 0.942
T= 10, p=64, dt=0.156: ratio 0.987
T= 20, p=64, dt=0.312: ratio 0.999


T= 40, p=64, dt=0.625: ratio 1.000


T= 80, p=64, dt=1.250: ratio 0.408


T= 80, p=128, dt=0.625: ratio 1.000


## Step 6: sample, evaluate classically, report the best string

In [7]:
sim = AerSimulator()
for p in [4, 64]:
    qc = qaoa_circuit(p, 40, measure=True)
    counts = sim.run(transpile(qc, sim), shots=500).result().get_counts()
    scored = sorted(((cut_weight(np.array([int(ch) for ch in s])), s, k) for s, k in counts.items()), reverse=True)
    best_cut, best_s, _ = scored[0]
    mean_cut = sum(cw * k for cw, s, k in scored) / 500
    print(f"p={p}: best sample {best_s} with cut {best_cut} (max {C_max}); mean over shots {mean_cut:.2f}")
    assert best_cut == C_max

p=4: best sample 10101 with cut 10.0 (max 10.0); mean over shots 5.38


p=64: best sample 10101 with cut 10.0 (max 10.0); mean over shots 10.00


The best sample reaches the optimum already at $p=4$ with 500 shots, while the mean
is far below it: reporting the expectation value understates what the algorithm
delivers, which is the book's point in step 6. The sign flip is the one place to get
wrong; without it the schedule converges on the *minimum* cut of weight 0.

In [8]:
# without the sign flip: minimum cut instead
zz0, z0, const0 = ising_terms(Q, c)
terms0 = [(label([(i, "Z"), (j, "Z")]), v) for (i, j), v in zz0.items()] + [(label([(i, "Z")]), v) for i, v in z0.items()]
qc = QuantumCircuit(n); qc.h(q); dt = 40 / 64
for j in range(1, 65):
    for P, coef in terms0: pauli_evolution(qc, P, coef * (j / 64) * dt, q)
    qc.rx(-2 * (1 - j / 64) * dt, q)
probs = Statevector(qc).probabilities(q[::-1])
print("unflipped sign, most likely string:", "".join(map(str, strings[int(np.argmax(probs))])), "cut =", cut_weight(strings[int(np.argmax(probs))]))

unflipped sign, most likely string: 00000 cut = 0.0
